In [14]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from meteor import MeteorInterface

In [ ]:
# FASTMIP Phase 1 ESMs:
ESMs = ['ACCESS-ESM1-5']#, 'CMCC-CM2-SR5', 'CNRM-CM6-1', 'CanESM5', 'EC-Earth3', 'INM-CM5-0', 'IPSL-CM6A-LR', 'MIROC-ES2L', 'MIROC6', 'MPI-ESM1-2-LR', 'MPI-ESM1-2-HR', 'MRI-ESM2-0']


In [3]:
# FASTMIP Phase 1 output to generate:
# Up to 2100
# 2.5x2.5 degree common grid
# 10 member ensemble for each ESM and scenario
# Annual mean tas (gridded)
# Monthly mean tas and pr (gridded)
# Both METEOR raw output and scaled to provided GSAT timeseries
# output filename convention for FASTMIP phase 1: emulations_{tas, pr}_{ann, mon}_METEOR_{esm}_{scenario}_g025.nc 

scenarios = ['ssp119']#, 'ssp126', 'ssp245', 'ssp370', 'ssp585']
FAIRpercentiles = ['5', '10', '50', '90', '95']
start_year = 2020
end_year = 2100
n_members = 10
output_dir = '../data/FASTMIP_phase1/METEOR_emulations/'
scaling_dir = '../data/FASTMIP_phase1/FAIR_GSAT/'

def format_FAIRssp(scenario):
    return f"SSP{scenario[3]}-{scenario[4:]}"

In [ ]:
# make hack to get the FAIR GSAT timeseries into the right format for scaling

In [10]:
# Create emulators for list of ESMs (uses cached models if available)
for esm in ESMs:
    print(f"Creating emulator for {esm}...")
    emulator = MeteorInterface(
        model=esm,
        variables=['tas', 'pr'],
        cache_dir='../cache'
    )
    print(f"Emulator created for {emulator.model} with variables: {emulator.variables}")
    
    print(f"Training emulator for {emulator.model}...")
    emulator.train(verbose=False)

    for scenario in scenarios:
        print(f"Emulating scenario {scenario}...")

        # Emulate gridded outputs for each scenario and save:
        ensemble_gridded = emulator.generate_ensemble_outputs(
            scenario=scenario,
            start_year=start_year,
            end_year=end_year,
            n_realizations=n_members,
            timeseries=['global'], 
            gridded={
                'annual': list(range(start_year, end_year + 1)),  # Annual mean grids for all years 
                'monthly': list(range(start_year, end_year + 1)),  # Monthly mean grids for all years
            },
            save_to=f"{output_dir}METEOR_{esm}_{scenario}.nc"
        )

        # Scale output to FAIR GSAT:
        # (Could also loop through percentiles here, for now only using 50th)
        for percentile in ['50']: #FAIRpercentiles:    
            scaling_ts = xr.open_dataset(f"{scaling_dir}GSAT_SSPmarker_{format_FAIRssp(scenario)}.nc")['tas'].sel(percentile=percentile, SCM='FaIRv1.6.2', time=slice(start_year, end_year)).rename({'time': 'year'})

            ensemble_scaled = emulator.generate_ensemble_outputs(
                scenario=scenario,
                start_year=start_year,
                end_year=end_year,
                n_realizations=n_members,
                timeseries=['global'], 
                gridded={
                    'annual': list(range(start_year, end_year + 1)),  # Annual mean grids for all years
                    'monthly': list(range(start_year, end_year + 1)),  # Monthly mean grids for all years 
                },
                save_to=f"{output_dir}METEOR_{esm}_{scenario}_scaledtoFAIR_{percentile}percentile.nc",
                temp_scaling_ts=scaling_ts
            )   
        

Creating emulator for ACCESS-ESM1-5...
Emulator created for ACCESS-ESM1-5 with variables: ['tas', 'pr']
Training emulator for ACCESS-ESM1-5...
🔧 Preparing pattern scaling training data for ACCESS-ESM1-5...
   ✅ Training data prepared for experiments:  ['base', 'co2x4', 'ssp245', 'sulxanom']
📥 Loading CICERO-SCM forcing data for ssp245...
   ✅ Loaded 801 concentration records
   ✅ Loaded 351 emission records
   ✅ Config: 1750-2100, emissions start: 1850
📦 Loading cached pattern scaling model from ../cache/pattern_scaling/cmip6-ACCESS-ESM1-5-aer-tas_pattern_scaling.pkl
✅ Pattern scaling model loaded from ../cache/pattern_scaling/cmip6-ACCESS-ESM1-5-aer-tas_pattern_scaling.pkl
Model loaded from ../cache/noise_models/ACCESS-ESM1-5_tas_noise_model.pkl
🔧 Preparing pattern scaling training data for ACCESS-ESM1-5...
   ✅ Training data prepared for experiments:  ['base', 'co2x4', 'ssp245', 'sulxanom']
📥 Loading CICERO-SCM forcing data for ssp245...
   ✅ Loaded 801 concentration records
   ✅ Loa

ValueError: temp_scaling_ts temporal extent (81) must match annual_prediction time dimension (351)

In [5]:
xr.open_dataset(f"{output_dir}METEOR_ACCESS-ESM1-5_ssp119.nc")

<xarray.Dataset> Size: 5GB
Dimensions:           (year: 81, realization: 10, lat: 145, lon: 192, date: 972)
Dimensions without coordinates: year, realization, lat, lon, date
Data variables:
    tas_grid_annual   (year, realization, lat, lon) float64 180MB ...
    tas_grid_monthly  (date, realization, lat, lon) float64 2GB ...
    tas_global        (realization, date) float64 78kB ...
    pr_grid_annual    (year, realization, lat, lon) float64 180MB ...
    pr_grid_monthly   (date, realization, lat, lon) float64 2GB ...
    pr_global         (realization, date) float64 78kB ...
Attributes:
    model:           ACCESS-ESM1-5
    scenario:        ssp119
    year_range:      2020-2100
    n_realizations:  10

In [ ]:
# work on solution for saving to netcdf

def test_to_netcdf(ensemble_gridded, output_path):
    datasets = {}

    
    for var_name, var_data in ensemble_gridded.variables.items():
        ds = xr.Dataset()
        print(var_name)

        # Do gridded variables first, because these actually return the years as coordinates, whereas the timeseries variables just return arrays without start/end year information
   
        for grid_name, grid_array in var_data.gridded.items():
            print(grid_name)
            safe_name = str(grid_name).replace("-", "_").replace(":", "_")
            if isinstance(grid_array, dict) and grid_name == "annual":
                    years = sorted(grid_array.keys())
                    stacked = xr.concat(
                        [grid_array[y] for y in years],
                        dim="year"
                    )
                    stacked = stacked.assign_coords(year=years)
                    ds[f"{var_name}_grid_{safe_name}"] = stacked

            elif isinstance(grid_array, dict) and grid_name == "monthly":
                    years = sorted(grid_array.keys())
                    stacked = xr.concat(
                                [grid_array[y] for y in years],
                                dim="year").assign_coords(year=years)
                    stacked = stacked.stack(date=("year", "month"))
                    date_vals = [y * 100 + m
                                 for y in years
                                 for m in range(1, 13)]
                    stacked = stacked.drop_vars(['date', 'year', 'month']).assign_coords(date=date_vals).transpose("date", "realization", "lat", "lon")
                    ds[f"{var_name}_grid_{safe_name}"] = stacked
    

        for ts_name, ts_array in var_data.timeseries.items():
            safe_name = ts_name.replace(":", "_").replace(".", "p")
            if ts_array.ndim == 1:
                dims = ("date",)
            elif ts_array.ndim == 2:
                dims = ("realization", "date")
            ds[f"{var_name}_{safe_name}"] = (dims, ts_array)
     
        datasets[var_name] = ds

    combined = xr.merge(list(datasets.values()))
    return combined